Libraries

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import dask.dataframe as dd
from IPython.display import display

pd.set_option('display.max_columns', None)

: 

In [ ]:
# Definir rango de fechas para Enero 2021 (no está disponible todo enero 2020)

start_date = '2021-01-01'
end_date = '2021-01-31'
# end_date = '2022-12-31'

date_range = pd.date_range(start=start_date, end=end_date)
# print(date_range)

df = []
df2 = []

In [ ]:
# Importar Enero 2021 - forma 1

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df.append(pd.read_csv(url))

# Combinar los DataFrames diarios en uno solo.
enero = pd.concat(df, ignore_index=True)

# Calcular tiempo de carga
load_time_1 = time.time() - start_time
print(f"Tiempo de carga: {load_time_1:.2f} s")

In [ ]:
## Instalar aiohttp (necesario para fsspec HTTPFileSystem usado por Dask/pandas)
#%pip install aiohttp -q
#
#import aiohttp

In [ ]:
# Importar Enero 2021 - forma 2 (con Dask)

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df2.append(dd.read_csv(url, dtype={'Admin2': 'object'}))

# Combinar los DataFrames diarios en uno solo.
enero = dd.concat(df2, ignore_index=True)

# Calcular tiempo de carga
load_time_2 = time.time() - start_time
print(f"Tiempo de carga con Dask: {load_time_2:.2f} s")

In [ ]:
# 1. Cargar y visualizar los primeros 5 registros

enero.head()

In [ ]:
# 2. Mostrar el número total de filas y columnas del DataFrame.

print('Filas en total: ', len(enero))
print('Columnas en total: ', len(enero.columns))

In [ ]:
# 3. Describir los tipos de datos (dtypes) y convertir las columnas necesarias (por ejemplo,
# fechas).

enero.dtypes

In [ ]:
# 3

# Formatear la columna Last_Update a tipo datetime

enero = enero.assign(Last_Update=dd.to_datetime(enero['Last_Update'], errors='coerce'))

enero.dtypes

In [ ]:
# 3

# Uso de memoria antes de conversión de tipos

enero_pd = enero.compute()

# Verificar memoria usada por el DataFrame pandas resultante
memoria = enero_pd.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memoria usada del DataFrame: {memoria:.2f} MB")

In [ ]:
# 3

# Uso de memoria después de conversión de tipos

enero['Province_State'] = enero['Province_State'].astype('category')
enero['Country_Region'] = enero['Country_Region'].astype('category')
enero['Combined_Key'] = enero['Combined_Key'].astype('category')

enero_pdf = enero.compute()

# Verificar memoria usada por el DataFrame pandas resultante
memoria_optimizacion = enero_pdf.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memoria usada del DataFrame tras optimización: {memoria_optimizacion:.2f} MB")

In [ ]:
print(f"Diferencia en uso de memoria: {(memoria - memoria_optimizacion):.2f} MB")

In [ ]:
print(f"Diferencia en uso de memoria: {(-(memoria - memoria_optimizacion) * 100 / memoria):.2f}%")

In [ ]:
# 4. Detectar y mostrar valores nulos o faltantes por columna.

display(enero.isnull().sum())

In [ ]:
# 5. Eliminar columnas irrelevantes (por ejemplo, códigos FIPS o coordenadas si no se usarán).

enero = enero.drop(columns=['FIPS', 'Admin2', 'Lat', 'Long_', 'Combined_Key'])
enero.head(0)

: 

In [ ]:
# 6. Estandarizar nombres de columnas (usar formato snake_case).

enero.columns = enero.columns.str.lower().str.replace(' ', '_')
enero.head(0)

In [ ]:
# 7. Homogeneizar nombres de países (ej. “US” → “United States”).

enero['country_region'] = enero['country_region'].replace({'US': 'United States'})
enero[enero['country_region'] == 'United States'].head(1)

In [ ]:
# 8. Convertir la columna last_update al formato YYYY-MM-DD (día preciso)
# Asegurar datetime y mantener sólo fecha (YYYY-MM-DD)
enero['last_update'] = dd.to_datetime(enero['last_update'], errors='coerce').dt.date
enero.head(1)

In [ ]:
# 9. Crear una columna active_cases = Confirmed - Deaths - Recovered.

enero['active_cases'] = (enero['confirmed'] - enero['deaths'] - enero['recovered'])
enero.head(1)

In [ ]:
# 10. Guardar el DataFrame limpio como covid_clean_enero2021.csv e indicar su tamaño en MB.

try:
    enero.compute().to_csv('covid_clean_enero2021.csv')
except:
    os.remove('covid_clean_enero2021.csv')
    enero.compute().to_csv('covid_clean_enero2021.csv')

file_size = os.path.getsize('covid_clean_enero2021.csv') / (1024 * 1024)  # Convertir a MB
print(f'El tamaño del archivo covid_clean_enero2021.csv es: {file_size:.2f} MB')